In [1]:
from pathlib import Path

ROOT = Path.cwd()

if ROOT.name == "notebooks":
    ROOT = ROOT.parent

print("Корень проекта:", ROOT)
print("src существует:", (ROOT / "src").exists())
print("data существует:", (ROOT / "data").exists())


Корень проекта: c:\Users\danmo\DTP\road-accidents-analysis
src существует: True
data существует: True


In [2]:
import sys

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.cleaning import (
    clean_dataset,
    replace_missing_values,
    convert_data_types,
    parse_datetime,
    add_time_features,
    decode,
    coords,
    merge_tables,
    save_processed
)

print("Импорт прошёл успешно!")

Импорт прошёл успешно!


In [3]:
import pandas as pd

data_path = ROOT / "data" / "raw"

dfcol = pd.read_csv(data_path / "dft-road-casualty-statistics-collision-last-5-years.csv")
dfveh = pd.read_csv(data_path / "dft-road-casualty-statistics-vehicle-last-5-years.csv")
dfcas = pd.read_csv(data_path / "dft-road-casualty-statistics-casualty-last-5-years.csv")

print("Collision:", dfcol.shape)
print("Vehicle:", dfveh.shape)
print("Casualty:", dfcas.shape)

C:\Users\danmo\AppData\Local\Temp\ipykernel_21416\4000243588.py:5: DtypeWarning: Columns (0: collision_index, 2: collision_ref_no) have mixed types. Specify dtype option on import or set low_memory=False.
  dfcol = pd.read_csv(data_path / "dft-road-casualty-statistics-collision-last-5-years.csv")


Collision: (513801, 44)
Vehicle: (937265, 32)
Casualty: (652821, 24)


C:\Users\danmo\AppData\Local\Temp\ipykernel_21416\4000243588.py:6: DtypeWarning: Columns (27: generic_make_model) have mixed types. Specify dtype option on import or set low_memory=False.
  dfveh = pd.read_csv(data_path / "dft-road-casualty-statistics-vehicle-last-5-years.csv")


In [4]:
df = clean_dataset(dfcol, dfveh, dfcas)

display(df.head())
display(df.info())

,collision_index,collision_year,collision_ref_no,location_easting_osgr,location_northing_osgr,longitude,latitude,police_force,collision_severity,number_of_vehicles,...,weekday_pd,is_weekend,part_of_day,has_coords,driver_age_min,driver_age_max,driver_age_mean,casualty_age_min,casualty_age_max,casualty_age_mean
0,202417H105824,2024,17H105824,448628.0,533968.0,-1.247,54.6984,17,Serious,2,...,3,False,day,True,33.0,66.0,49.5,33.0,33.0,33.000000
1,202517M115025,2025,17M115025,452379.0,519038.0,-1.19147,54.56387,17,Serious,2,...,5,True,day,True,17.0,40.0,28.5,17.0,17.0,17.000000
2,202517L306225,2025,17L306225,459273.0,523811.0,-1.0839,54.606,17,Slight,2,...,6,True,evening,True,35.0,53.0,44.0,10.0,35.0,19.666667
3,202417S104724,2024,17S104724,440460.0,517548.0,-1.37599,54.55157,17,Slight,2,...,5,True,day,True,25.0,45.0,35.0,22.0,25.0,23.500000
4,202517S106525,2025,17S106525,444331.0,518776.0,-1.31596,54.56228,17,Serious,2,...,4,False,morning,True,16.0,40.0,28.0,16.0,16.0,16.000000


<class 'pandas.DataFrame'>
RangeIndex: 513801 entries, 0 to 513800
Data columns (total 58 columns):
 #   Column                                            Non-Null Count   Dtype         
---  ------                                            --------------   -----         
 0   collision_index                                   513801 non-null  string        
 1   collision_year                                    513801 non-null  Int64         
 2   collision_ref_no                                  513801 non-null  string        
 3   location_easting_osgr                             513748 non-null  Float64       
 4   location_northing_osgr                            513748 non-null  Float64       
 5   longitude                                         513748 non-null  Float64       
 6   latitude                                          513748 non-null  Float64       
 7   police_force                                      513801 non-null  Int64         
 8   collision_severity       

None

In [5]:
print("Размер итоговой таблицы:", df.shape)

print("Количество уникальных ДТП:", df["collision_index"].nunique())

print("Дубликаты:", df.duplicated().sum())

print("Пропуски:")
display(
    df.isna()
    .sum()
    .sort_values(ascending=False)
    .head(15)
)

print("Наличие координат:")
display(df["has_coords"].value_counts(dropna=False))

print("Заполнено datetime:", df["datetime"].notna().sum())

display(
    df[["date", "time", "datetime", "year", "month", "hour", "part_of_day"]].head(10)
)

Размер итоговой таблицы: (513801, 58)
Количество уникальных ДТП: 513801
Дубликаты: 0
Пропуски:


local_authority_district                            513607
junction_control                                    217897
enhanced_severity_collision                         204752
second_road_number                                  178098
special_conditions_at_site                          151370
carriageway_hazards_historic                        151367
pedestrian_crossing_human_control_historic          150889
pedestrian_crossing_physical_facilities_historic    150860
junction_detail_historic                            147476
trunk_road_flag                                      36318
driver_age_min                                       27640
driver_age_mean                                      27640
driver_age_max                                       27640
junction_detail                                      19982
second_road_class                                    11566
dtype: int64

Наличие координат:


has_coords
True     513748
False        53
Name: count, dtype: int64

Заполнено datetime: 513801


,date,time,datetime,year,month,hour,part_of_day
0,2024-10-03,16:06,2024-10-03 16:06:00,2024,10,16,day
1,2025-11-29,16:34,2025-11-29 16:34:00,2025,11,16,day
2,2025-06-01,22:07,2025-06-01 22:07:00,2025,6,22,evening
3,2024-04-20,14:40,2024-04-20 14:40:00,2024,4,14,day
4,2025-05-16,07:25,2025-05-16 07:25:00,2025,5,7,morning
5,2025-06-22,13:30,2025-06-22 13:30:00,2025,6,13,day
6,2025-07-13,15:10,2025-07-13 15:10:00,2025,7,15,day
7,2024-05-13,12:10,2024-05-13 12:10:00,2024,5,12,day
8,2024-06-20,16:30,2024-06-20 16:30:00,2024,6,16,day
9,2025-04-30,16:31,2025-04-30 16:31:00,2025,4,16,day


In [16]:
save_processed(df, ROOT / "data" / "processed" / "collisions.parquet")

WindowsPath('c:/Users/danmo/DTP/road-accidents-analysis/data/processed/collisions.parquet')

In [6]:
tables = {
    "collision": dfcol,
    "vehicle": dfveh,
    "casualty": dfcas
}

for name, df in tables.items():
    print(f"\n========== {name.upper()} ==========")
    print("Размер:", df.shape)
    print("Колонки:")
    print(df.columns.tolist())


========== COLLISION ==========
Размер: (513801, 44)
Колонки:
['collision_index', 'collision_year', 'collision_ref_no', 'location_easting_osgr', 'location_northing_osgr', 'longitude', 'latitude', 'police_force', 'collision_severity', 'number_of_vehicles', 'number_of_casualties', 'date', 'day_of_week', 'time', 'local_authority_district', 'local_authority_ons_district', 'local_authority_highway', 'local_authority_highway_current', 'first_road_class', 'first_road_number', 'road_type', 'speed_limit', 'junction_detail_historic', 'junction_detail', 'junction_control', 'second_road_class', 'second_road_number', 'pedestrian_crossing_human_control_historic', 'pedestrian_crossing_physical_facilities_historic', 'pedestrian_crossing', 'light_conditions', 'weather_conditions', 'road_surface_conditions', 'special_conditions_at_site', 'carriageway_hazards_historic', 'carriageway_hazards', 'urban_or_rural_area', 'did_police_officer_attend_scene_of_accident', 'trunk_road_flag', 'lsoa_of_accident_locat

In [7]:
for name, df in tables.items():
    print(f"\n========== {name.upper()} ==========")
    display(df.dtypes.to_frame("dtype"))


========== COLLISION ==========


,dtype
collision_index,object
collision_year,int64
collision_ref_no,object
location_easting_osgr,float64
location_northing_osgr,float64
longitude,float64
latitude,float64
police_force,int64
collision_severity,int64
number_of_vehicles,int64



========== VEHICLE ==========


,dtype
collision_index,str
collision_year,int64
collision_ref_no,str
vehicle_reference,int64
vehicle_type,int64
towing_and_articulation,int64
vehicle_manoeuvre_historic,int64
vehicle_manoeuvre,int64
vehicle_direction_from,int64
vehicle_direction_to,int64



========== CASUALTY ==========


,dtype
collision_index,str
collision_year,int64
collision_ref_no,str
vehicle_reference,int64
casualty_reference,int64
casualty_class,int64
sex_of_casualty,int64
age_of_casualty,int64
age_band_of_casualty,int64
casualty_severity,int64


In [8]:
print("COLLISIONS")
print("Дубликаты collision_index:",
      dfcol["collision_index"].duplicated().sum())
print("Пропуски collision_index:",
      dfcol["collision_index"].isna().sum())

print("\nVEHICLES")
print("Дубликаты составного ключа:",
      dfveh.duplicated(
          subset=["collision_index", "vehicle_reference"]
      ).sum())

print("\nCASUALTIES")
print("Дубликаты составного ключа:",
      dfcas.duplicated(
          subset=["collision_index", "casualty_reference"]
      ).sum())

COLLISIONS
Дубликаты collision_index: 0
Пропуски collision_index: 0

VEHICLES
Дубликаты составного ключа: 0

CASUALTIES
Дубликаты составного ключа: 0
